In [ ]:
import sys
import random
import os
import glob
import pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from astropy.io import fits
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.colors as colors

In [4]:
# # Eliminador

# pickle_folder = 'extra'
# spectrums_folder = r'spectrumsstars'

# # Obtener la lista de archivos pickle que comienzan con "batch_files_list" y terminan con ".pkl"
# pickle_files = [f for f in os.listdir(pickle_folder) if f.startswith('batch_files_list') and f.endswith('.pkl')]

# # Recorrer cada archivo pickle
# for pickle_file in pickle_files:
#     pickle_path = os.path.join(pickle_folder, pickle_file)
#     print(f"Procesando {pickle_path}...")
#     try:
#         with open(pickle_path, 'rb') as f:
#             file_list = pickle.load(f)
#     except Exception as e:
#         print(f"Error abriendo {pickle_path}: {e}")
#         continue
#     # Recorrer la lista de nombres y eliminar los archivos correspondientes en la carpeta spectrums
#     for filename in file_list:
#         file_path = os.path.join(spectrums_folder, filename)
#         if os.path.exists(file_path):
#             try:
#                 os.remove(file_path)
#                 print(f"Eliminado: {file_path}")
#             except Exception as e:
#                 print(f"Error eliminando {file_path}: {e}")
#         else:
#             print(f"Archivo no encontrado: {file_path}")

# print("Proceso de eliminación completado.")

In [ ]:
# Procesamiento de archivos completo

# APLICACIÓN A LOS ARCHIVOS DE LA CARPETA SPECTRUMS
folder_path = r'spectrumsstars'
batch_size = 10000  # Número de archivos a procesar por lote
index_file = "extra/batch_index.txt"

# Verificar que el archivo batch_index.txt exista
if not os.path.exists(index_file):
    print(f"Error: El archivo {index_file} no existe. Deteniendo la ejecución.")
    sys.exit(1)

# Intentar leer y convertir el contenido a un número entero
with open(index_file, "r") as f:
    content = f.read().strip()
    try:
        batch_index = int(content)
    except ValueError:
        print(f"Error: El archivo {index_file} no contiene un número entero válido. Deteniendo la ejecución.")
        sys.exit(1)

def expand_points(wavelength, flux, target_count=5000):
    # Convertir a listas para facilitar las inserciones
    wl = list(wavelength)
    fl = list(flux)
    
    # Calcular las diferencias absolutas entre puntos consecutivos
    diffs = [abs(fl[i+1] - fl[i]) for i in range(len(fl)-1)]
    # Obtener los índices ordenados de mayor a menor diferencia
    sorted_indices = sorted(range(len(diffs)), key=lambda i: diffs[i], reverse=True)
    
    # Insertar nuevos puntos utilizando los índices ordenados
    while len(wl) < target_count:
        # Se recorre la lista de índices en orden descendente para evitar problemas con el reordenamiento
        for idx in sorted_indices:
            if len(wl) >= target_count:
                break
            # Calcular la interpolación lineal entre el punto idx y el siguiente
            new_wl = (wl[idx] + wl[idx+1]) / 2
            new_fl = (fl[idx] + fl[idx+1]) / 2
            # Insertar el nuevo punto en la posición correspondiente
            wl.insert(idx+1, new_wl)
            fl.insert(idx+1, new_fl)
    
    return np.array(wl), np.array(fl)

# Obtener la lista inicial de archivos FITS
files = [f for f in os.listdir(folder_path) if f.endswith('.fits')]

while files:
    # Seleccionar aleatoriamente el lote actual (si quedan menos de batch_size, se toman todos)
    if len(files) < batch_size:
        current_batch = files.copy()
    else:
        current_batch = random.sample(files, batch_size)
    
    # Diccionario de almacenamiento para el lote actual
    spectra_data = {}
    i = 0

    for filename in current_batch:
        file_path = os.path.join(folder_path, filename)
        try:
            with fits.open(file_path) as hdul:
                # Verifica que el archivo tenga las extensiones esperadas
                if len(hdul) > 2:
                    flux_data = hdul[1].data["flux"]         # Datos de flujo
                    loglam_data = hdul[1].data["loglam"]        # Datos de log(lambda)
                    wavelength_data = 10 ** loglam_data         # Convertir log(lambda) a longitud de onda
                    redshift = hdul[2].data["Z"][0]             # Extraer redshift
                    
                    # Aplicar la interpolación para obtener 5000 puntos
                    interp_wavelength, interp_flux = expand_points(wavelength_data, flux_data, target_count=5000)
                    
                    # Guardar los datos
                    spectra_data[filename] = {
                        "wavelength": interp_wavelength,
                        "flux": interp_flux,
                        "redshift": redshift
                    }
                    
                    i += 1
                    if i % 1000 == 0:
                        print(f"Procesado {filename} ({i})")
        except Exception as e:
            print(f"Error procesando {filename}: {e}")
    
    print(f"Se procesaron {len(spectra_data)} archivos FITS en el lote {batch_index}.")
    
    # Guardar el diccionario en un archivo pickle
    output_file = f'data/spectra_data_complete{batch_index}.pkl'
    with open(output_file, 'wb') as f:
        pickle.dump(spectra_data, f)
    print(f"Datos guardados en {output_file}")
    
    # Guardar la lista de archivos procesados en este lote
    batch_list_file = f'extra/batch_files_list{batch_index}.pkl'
    with open(batch_list_file, 'wb') as f:
        pickle.dump(current_batch, f)
    print(f"Lista de archivos del lote {batch_index} guardada en {batch_list_file}")
    
    # Actualizar la lista de archivos: eliminar los procesados
    files = [f for f in files if f not in current_batch]
    
    # Actualizar el archivo batch_index.txt para la siguiente iteración
    batch_index += 1
    with open(index_file, "w") as f:
        f.write(str(batch_index))
    
print("Se han procesado y eliminado todos los archivos de la carpeta.")

Procesado spec-1888-53239-0116.fits (1000)
Procesado spec-0471-51924-0545.fits (2000)
Procesado spec-2945-54505-0096.fits (3000)
Procesado spec-3317-54908-0127.fits (4000)
Procesado spec-1079-52621-0380.fits (5000)
Procesado spec-3247-54888-0132.fits (6000)
Procesado spec-3265-54888-0397.fits (7000)
Procesado spec-1066-52589-0268.fits (8000)
Procesado spec-4642-55926-0420.fits (9000)
Procesado spec-5196-55831-0118.fits (10000)
Se procesaron 10000 archivos FITS en el lote 3415.
Datos guardados en data/spectra_data_complete3415.pkl
Lista de archivos del lote 3415 guardada en extra/batch_files_list3415.pkl
Procesado spec-2278-53711-0025.fits (1000)
Procesado spec-5954-56462-0619.fits (2000)
Procesado spec-3770-55234-0608.fits (3000)
Procesado spec-7393-56749-0890.fits (4000)
Procesado spec-1244-52674-0472.fits (5000)
Procesado spec-1096-52974-0181.fits (6000)
Procesado spec-11125-58433-0048.fits (7000)
Procesado spec-3777-55210-0808.fits (8000)
Procesado spec-2076-53442-0610.fits (9000)
P

In [ ]:
# # Buscar todos los archivos que comiencen con "spectra_data_complete" en la carpeta "data"
# data_files = glob.glob(os.path.join('data', 'spectra_data_100k*.pkl'))
# data_files = sorted(data_files)
# print(f"Se encontraron {len(data_files)} archivos pickle.")

# # Primer paso: contar el número total de espectros
# total_spectra = 0
# for file in data_files:
#     with open(file, 'rb') as f:
#         data_part = pickle.load(f)
#     total_spectra += len(data_part)
# print(f"Total de espectros: {total_spectra}")

# # Prealocar archivos memmap para flux, wavelength y redshift en la carpeta 'data'
# flux_mmap_path = os.path.join('data', 'flux_mmap.dat')
# wavelength_mmap_path = os.path.join('data', 'wavelength_mmap.dat')
# redshift_mmap_path = os.path.join('data', 'redshift_mmap.dat')

# num_points = 5000
# flux_mmap = np.memmap(flux_mmap_path, dtype='float32', mode='w+', shape=(total_spectra, num_points))
# wavelength_mmap = np.memmap(wavelength_mmap_path, dtype='float32', mode='w+', shape=(total_spectra, num_points))
# redshift_mmap = np.memmap(redshift_mmap_path, dtype='float32', mode='w+', shape=(total_spectra,))

# # Segundo pase: cargar datos de los archivos pickle y escribirlos en los memmaps
# current_index = 0
# for file in data_files:
#     with open(file, 'rb') as f:
#         data_part = pickle.load(f)
#     print(f"Procesando {file} con {len(data_part)} espectros...")
#     for key in data_part:
#         espectro = data_part[key]
#         flux = espectro["flux"]         # Array de 5000 puntos
#         wavelength = espectro["wavelength"]  # Array de 5000 puntos
#         redshift = espectro["redshift"]      # Valor escalar
        
#         # Almacenar en el memmap
#         flux_mmap[current_index, :] = flux
#         wavelength_mmap[current_index, :] = wavelength
#         redshift_mmap[current_index] = redshift
        
#         current_index += 1

# # Asegurar que todos los cambios se escriban en disco
# flux_mmap.flush()
# wavelength_mmap.flush()
# redshift_mmap.flush()

# print("Se han guardado los datos en archivos memmap dentro de la carpeta 'data'.")

In [ ]:
# import os
# import numpy as np

# # Parámetros de configuración
# current_dataset_dir = 'data'
# current_total = 100000  # Número de espectros
# num_points = 5000       # Número de puntos por espectro

# # Rutas de los archivos originales (1D)
# flux_path = os.path.join(current_dataset_dir, 'spectra_data_100k_flux_mmap.dat')
# wavelength_path = os.path.join(current_dataset_dir, 'spectra_data_100k_wavelength_mmap.dat')

# # Cargar los memmaps en modo lectura (sin especificar shape, se obtiene la forma raw)
# flux_memmap_raw = np.memmap(flux_path, dtype='float32', mode='r')
# wavelength_memmap_raw = np.memmap(wavelength_path, dtype='float32', mode='r')

# print("Flux raw shape:", flux_memmap_raw.shape)
# print("Wavelength raw shape:", wavelength_memmap_raw.shape)

# # Convertir a arrays 2D con shape (100000, 5000)
# flux_2d = flux_memmap_raw.reshape((current_total, num_points))
# wavelength_2d = wavelength_memmap_raw.reshape((current_total, num_points))

# print("Flux 2D shape:", flux_2d.shape)
# print("Wavelength 2D shape:", wavelength_2d.shape)

# # Opcional: guardar los nuevos arrays 2D en nuevos archivos memmap
# new_flux_path = os.path.join(current_dataset_dir, 'spectra_data_100k_flux_2d.dat')
# new_wavelength_path = os.path.join(current_dataset_dir, 'spectra_data_100k_wavelength_2d.dat')

# flux_memmap_2d = np.memmap(new_flux_path, dtype='float32', mode='w+', shape=(current_total, num_points))
# wavelength_memmap_2d = np.memmap(new_wavelength_path, dtype='float32', mode='w+', shape=(current_total, num_points))

# flux_memmap_2d[:] = flux_2d[:]
# wavelength_memmap_2d[:] = wavelength_2d[:]

# flux_memmap_2d.flush()
# wavelength_memmap_2d.flush()

# print("Nuevos archivos memmap 2D creados:")
# print(f"  Flux: {new_flux_path}")
# print(f"  Wavelength: {new_wavelength_path}")

In [ ]:
# # Función para interpolación adaptativa (15000 FITS = 5hrs POCO VIABLE EN DATASETS GRANDES PREPARAR LOS DATOS)

# def expand_points(wavelength, flux, target_count=5000):
#     # Convertimos a listas para facilitar las inserciones
#     wl = list(wavelength)
#     fl = list(flux)
    
#     # Continuamos insertando hasta alcanzar el número deseado de puntos
#     while len(wl) < target_count:
#         # Calcular las diferencias absolutas en flux entre puntos consecutivos
#         diffs = [abs(fl[i+1] - fl[i]) for i in range(len(fl) - 1)]
#         # Encontrar el índice donde la diferencia es máxima
#         max_idx = np.argmax(diffs)
#         # Interpolar linealmente para obtener un nuevo punto
#         new_wl = (wl[max_idx] + wl[max_idx+1]) / 2
#         new_fl = (fl[max_idx] + fl[max_idx+1]) / 2
#         # Insertar el nuevo punto en la posición correspondiente
#         wl.insert(max_idx+1, new_wl)
#         fl.insert(max_idx+1, new_fl)
        
#     return np.array(wl), np.array(fl)

# # TEST CON EL ANTERIOR FITS
# expanded_wavelength, expanded_flux = expand_points(test_wavelength, test_flux, target_count=5000)

# # Graficar el espectro ampliado
# plt.figure(figsize=(12, 6))
# plt.plot(expanded_wavelength, expanded_flux, label="Espectro Expandido")
# plt.xlabel("Longitud de onda (Ångstrom)")
# plt.ylabel("Flujo (10^-17 erg/s/cm²/Å)")
# plt.title("Espectro vs. Flujo (Expandido a 5000 puntos)")
# plt.legend()
# plt.grid()
# plt.tight_layout()
# plt.show()